# 🔬 Patent Analysis Notebook
This notebook analyzes patent data from a SQLite database and generates visualizations for:
- Top 10 Inventors by Patent Count
- Top 10 Companies by Patent Count
- US Patents Granted Per Year (1976–2025)

## 📦 1. Imports & Setup

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

DB_PATH = "data/patents.db"
REPORTS = "reports"
os.makedirs(REPORTS, exist_ok=True)

print(f"Database path : {DB_PATH}")
print(f"Reports folder: {REPORTS}")

## 🗄️ 2. Load Data from SQLite

In [ ]:
conn = sqlite3.connect(DB_PATH)
print("Connected to database.")

In [ ]:
# Top 10 Inventors
top_inventors = pd.read_sql("""
    SELECT name, COUNT(DISTINCT patent_id) AS patents
    FROM inventors WHERE name != ''
    GROUP BY inventor_id
    ORDER BY patents DESC LIMIT 10
""", conn)

print(f"Loaded {len(top_inventors)} inventors")
top_inventors

In [ ]:
# Top 10 Companies
top_companies = pd.read_sql("""
    SELECT name, COUNT(DISTINCT patent_id) AS patents
    FROM companies WHERE name IS NOT NULL
    GROUP BY company_id
    ORDER BY patents DESC LIMIT 10
""", conn)

print(f"Loaded {len(top_companies)} companies")
top_companies

In [ ]:
# Patents Per Year
patents_per_year = pd.read_sql("""
    SELECT year, COUNT(*) AS patents
    FROM patents WHERE year IS NOT NULL
    GROUP BY year ORDER BY year ASC
""", conn)

print(f"Loaded data for {len(patents_per_year)} years ({patents_per_year['year'].min()} – {patents_per_year['year'].max()})")
patents_per_year.head()

In [ ]:
conn.close()
print("Database connection closed.")

## 📊 3. Chart 1 — Top 10 Inventors by Patent Count

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(
    top_inventors["name"][::-1],
    top_inventors["patents"][::-1],
    color="steelblue"
)
ax.set_xlabel("Number of Patents")
ax.set_title("Top 10 Inventors by Patent Count")
plt.tight_layout()

out_path = f"{REPORTS}/top_inventors.png"
plt.savefig(out_path)
plt.show()
print(f"Saved → {out_path}")

## 🏢 4. Chart 2 — Top 10 Companies by Patent Count

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(
    top_companies["name"][::-1],
    top_companies["patents"][::-1],
    color="darkorange"
)
ax.set_xlabel("Number of Patents")
ax.set_title("Top 10 Companies by Patent Count")
plt.tight_layout()

out_path = f"{REPORTS}/top_companies.png"
plt.savefig(out_path)
plt.show()
print(f"Saved → {out_path}")

## 📈 5. Chart 3 — US Patents Granted Per Year

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    patents_per_year["year"],
    patents_per_year["patents"],
    color="green",
    linewidth=2
)
ax.fill_between(
    patents_per_year["year"],
    patents_per_year["patents"],
    alpha=0.2,
    color="green"
)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Patents")
ax.set_title("US Patents Granted Per Year (1976-2025)")
plt.tight_layout()

out_path = f"{REPORTS}/patents_per_year.png"
plt.savefig(out_path)
plt.show()
print(f"Saved → {out_path}")

## ✅ 6. Summary

In [ ]:
saved = [f for f in os.listdir(REPORTS) if f.endswith(".png")]
print(f"All charts saved to '{REPORTS}/'")
for f in sorted(saved):
    print(f"  ✔ {f}")